# 03 — Data Cleaning

**EduPredict — Student Exam Performance Prediction**

This notebook validates the actual dataset, removes the invalid target record from the modeling copy, audits missing values, saves the cleaned dataset, and creates the 80/20 train-test split.

**Important:** The raw CSV is never overwritten. Missing categorical values are handled later inside the fitted preprocessing pipeline to prevent data leakage.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
DATA_PATH = Path("../data/raw/StudentPerformanceFactors.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Original shape:", df.shape)

Dataset loaded successfully.
Original shape: (6607, 20)


In [3]:
print("Number of duplicate rows:", df.duplicated().sum())

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))

print("\nTarget statistics:")
print(df["Exam_Score"].describe())

Number of duplicate rows: 0

Missing values:


Parental_Education_Level      90
Teacher_Quality               78
Distance_from_Home            67
Hours_Studied                  0
Attendance                     0
Gender                         0
Learning_Disabilities          0
Physical_Activity              0
Peer_Influence                 0
School_Type                    0
Family_Income                  0
Tutoring_Sessions              0
Internet_Access                0
Motivation_Level               0
Previous_Scores                0
Sleep_Hours                    0
Extracurricular_Activities     0
Access_to_Resources            0
Parental_Involvement           0
Exam_Score                     0
dtype: int64


Target statistics:
count    6607.000000
mean       67.235659
std         3.890456
min        55.000000
25%        65.000000
50%        67.000000
75%        69.000000
max       101.000000
Name: Exam_Score, dtype: float64


In [4]:
invalid_target_mask = ~df["Exam_Score"].between(0, 100)

print("Invalid Exam_Score records:", invalid_target_mask.sum())

if invalid_target_mask.sum() > 0:
    display(df.loc[invalid_target_mask, ["Exam_Score"]])

Invalid Exam_Score records: 1


,Exam_Score
1525,101


In [5]:
df_clean = df.loc[~invalid_target_mask].copy()

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Rows removed:", len(df) - len(df_clean))

Original shape: (6607, 20)
Cleaned shape: (6606, 20)
Rows removed: 1


In [6]:
print("Minimum Exam_Score:", df_clean["Exam_Score"].min())
print("Maximum Exam_Score:", df_clean["Exam_Score"].max())
print("Remaining invalid target values:",
      (~df_clean["Exam_Score"].between(0, 100)).sum())

Minimum Exam_Score: 55
Maximum Exam_Score: 100
Remaining invalid target values: 0


In [7]:
duplicate_count = df_clean.duplicated().sum()
print("Duplicate rows after cleaning:", duplicate_count)

Duplicate rows after cleaning: 0


In [8]:
missing_summary = pd.DataFrame({
    "Missing_Count": df_clean.isnull().sum(),
    "Missing_Percentage": (
        df_clean.isnull().sum() / len(df_clean) * 100
    ).round(2)
})

missing_summary = (
    missing_summary[missing_summary["Missing_Count"] > 0]
    .sort_values("Missing_Count", ascending=False)
)

display(missing_summary)

,Missing_Count,Missing_Percentage
Parental_Education_Level,90,1.36
Teacher_Quality,78,1.18
Distance_from_Home,67,1.01


In [9]:
rows_with_missing = df_clean.isnull().any(axis=1)

print("Rows containing at least one missing value:", rows_with_missing.sum())
print("Total missing cells:", df_clean.isnull().sum().sum())

Rows containing at least one missing value: 229
Total missing cells: 235


In [10]:
missing_cols = [
    "Parental_Education_Level",
    "Teacher_Quality",
    "Distance_from_Home"
]

missing_per_row = df_clean[missing_cols].isnull().sum(axis=1)
print(missing_per_row.value_counts().sort_index())

0    6377
1     223
2       6
Name: count, dtype: int64


## Missing-Value Treatment Decision

The missing categorical values are **not manually filled** in the raw or cleaned CSV.

They will be handled later inside the machine-learning preprocessing pipeline using:

```python
SimpleImputer(strategy="most_frequent")
```

The imputer will be fitted only on the training data and then applied to validation/test data. This prevents preprocessing leakage.

In [11]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_PATH = PROCESSED_DIR / "student_performance_clean.csv"
df_clean.to_csv(CLEAN_PATH, index=False)

print("Cleaned dataset saved to:")
print(CLEAN_PATH)
print("Saved shape:", df_clean.shape)

Cleaned dataset saved to:
../data/processed/student_performance_clean.csv
Saved shape: (6606, 20)


In [12]:
X = df_clean.drop(columns=["Exam_Score"])
y = df_clean["Exam_Score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5284, 19)
X_test : (1322, 19)
y_train: (5284,)
y_test : (1322,)


In [13]:
print("Total rows:", len(X_train) + len(X_test))
print("Expected rows:", len(df_clean))
print("Train + Test correct:", len(X_train) + len(X_test) == len(df_clean))
print("Target missing in train:", y_train.isnull().sum())
print("Target missing in test:", y_test.isnull().sum())

Total rows: 6606
Expected rows: 6606
Train + Test correct: True
Target missing in train: 0
Target missing in test: 0


# Data Cleaning Summary

| Check | Result |
|---|---:|
| Original records | 6,607 |
| Original columns | 20 |
| Invalid `Exam_Score` records | 1 |
| Records used for modeling | 6,606 |
| Duplicate records | 0 |
| Missing cells | 235 |
| Rows affected by missing values | 229 |
| Training records | 5,284 |
| Test records | 1,322 |

### Cleaning decisions

1. The original raw CSV is preserved unchanged.
2. The single record with `Exam_Score = 101` is excluded from the modeling dataset because it lies outside the expected 0–100 target range.
3. Missing categorical values are retained at this stage.
4. Missing-value imputation is performed later inside the fitted preprocessing pipeline using training data only.
5. The modeling dataset is split into 80% training and 20% test data using `random_state=42`.